# Where does the lead slope live? Circulations vs boundary reads by colour

## Question
On the square lattice, $1/c = 1.36 + 0.70 \times \text{lead}$. Off the square lattice, the slope $b$ falls with the loop length $\ell$ that the lead letter closes:
0.86 (hex R, $\ell=3$), 0.70 (square, 4), 0.34 (triangular, 6), 0.28 (hex r, 6).
The first-principles reading $b = \ell_b / N_b$ (boundary cells per lap over births per circulation episode) had never been measured.
This experiment measures the two factors of $b$ and tests whether circulation frequency carries the drop across lattices.

## Assumptions and definitions
- Everything as in `experiments/archive/2026-10-lattice-c-law` (octave window $k=26$, steps $(2^{26}, 2^{27}]$).
- **Return**: a step starting on a cell last read exactly $\ell$ steps earlier (square 4, triangular 6, hex r 6, hex R 3).
- **Circulation**: maximal run of consecutive returns of length $\ge \ell$.
  - $\epsilon$: circulations per birth
  - $R$: mean run length of circulations (capped at 63)
  - $C$: total steps in circulation runs per birth ($= \epsilon R$)
  - $C_b$: boundary steps in circulation runs per birth
- Gates: **T2** requires births and boundary steps to match parent `ants.csv` exactly, with $\sum_k \text{edge}_k = \text{boundary steps}$.

In [1]:
import hashlib, json, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
R = ROOT / "experiments/archive/2026-10-lattice-c-law-colours/results"
for n in ["traces.jsonl", "analysis.json", "analysis.txt", "explore.txt", "run.log"]:
    print(hashlib.sha256((R / n).read_bytes()).hexdigest()[:16], n)
git_rev = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print("git", git_rev or "unresolvable")
A = json.loads((R / "analysis.json").read_text())

f848a4d5e1d4a4c4 traces.jsonl
fd85474a68b2aad0 analysis.json
8ed369d97dc7b293 analysis.txt
73bc6af73b260eb3 explore.txt
4a1c16fdf8825f98 run.log
git 44ed81f


## Results
### Gate T2 and Registered Predictions (P1–P6)

In [2]:
# Summary of registered predictions and outcomes
for stratum, data in A.items():
    print(f"=== {stratum} (ell={data['ell']}) ===")
    print(f"  inv_c medians: {[round(x, 3) for x in data['inv_c_meds']]}")
    print(f"  Cb medians:    {[round(x, 3) for x in data['Cb_meds']]}")
    print(f"  C medians:     {[round(x, 1) for x in data['C_meds']]}")
    print(f"  b_med: {data['b_med']:.3f} | Theil-Sen b: {data['b_ts']:.3f} | Cb slope: {data['Cb_slope']:.3f} | R slope: {data['R_slope']:.3f} (R slope/ell: {data['R_slope']/data['ell']:.3f})")
    print(f"  eps(lead 3): {data['eps3']:.2f}")

=== square/R (ell=4) ===
  inv_c medians: [2.087, 2.616, 3.361, 4.102, 4.817]
  Cb medians:    [0.018, 0.313, 0.372, 1.127, 1.08]
  C medians:     [32.3, 29.5, 50.1, 70.3, 118.5]
  b_med: 0.695 | Theil-Sen b: 0.691 | Cb slope: 0.294 | R slope: 0.330 (R slope/ell: 0.082)
  eps(lead 3): 9.60
=== tri/R (ell=6) ===
  inv_c medians: [1.604, 1.971, 2.381, 2.647, 2.942]
  Cb medians:    [0.0, 0.004, 0.047, 0.123, 0.264]
  C medians:     [4.1, 4.1, 10.8, 13.2, 27.7]
  b_med: 0.335 | Theil-Sen b: 0.347 | Cb slope: 0.065 | R slope: 0.403 (R slope/ell: 0.067)
  eps(lead 3): 1.33
=== hex/R (ell=3) ===
  inv_c medians: [3.591, 4.245, 5.414, 6.093, 7.039]
  Cb medians:    [0.024, 0.207, 0.367, 0.611, 1.089]
  C medians:     [8.3, 20.4, 30.6, 71.8, 112.6]
  b_med: 0.875 | Theil-Sen b: 0.850 | Cb slope: 0.253 | R slope: 0.598 (R slope/ell: 0.199)
  eps(lead 3): 7.80
=== hex/r (ell=6) ===
  inv_c medians: [2.629, 2.768, 3.041, 3.39, 3.717]
  Cb medians:    [0.0, 0.003, 0.037, 0.129, 0.181]
  C medians:

### Why P3 is not evidence
The registered discriminator P3 (comparing $\epsilon$ across lattices) appeared to pass formally with huge ratios (7.2 and 75).
However, the circulation definition counted interior returns as well as boundary returns.
$C$ (steps in circulation runs) is 30–120 per birth on the square lattice while $1/c$ is only 2–5!
Interior loops completely swamped the boundary episodes.
The boundary-only version $C_b$ carries only 17–42% of the slope ($C_b$ slope / $b_{\text{med}} = 0.42$ square, $0.19$ tri, $0.29$ hex R, $0.17$ hex r), so P1 failed.
The circulation-lap decomposition $b = \ell_b / N_b$ was therefore **not validly measured** by this instrument.

### Post hoc: Where does the slope live? By colour!

In [3]:
# Post hoc decomposition of boundary reads by colour group
print((R / "explore.txt").read_text())

non-escaped rules with births>0: 854 (NOT the kept set: no blob filter; medians below are robust to the odd rule)

== square/R  ell=4  n=208
lead  n  | per-birth boundary reads by colour group: c0   circ(1..m-1)  kill(m)  later(>m) | total | lag-ell share of circ reads | lag-ell share of all colour>=1 reads
  1  47  |  0.982    0.000    0.739    0.360 |  2.079 |    nan |  0.577
  2  37  |  0.999    0.853    0.537    0.179 |  2.591 |  0.872 |  0.668
  3  41  |  0.991    1.636    0.534    0.147 |  3.346 |  0.671 |  0.567
  4  42  |  0.990    2.424    0.506    0.131 |  4.084 |  0.727 |  0.662
  5  41  |  0.992    3.192    0.489    0.092 |  4.762 |  0.659 |  0.605
   slope of median c0    vs lead:   0.001
   slope of median circ  vs lead:   0.796
   slope of median kill  vs lead:  -0.053
   slope of median later vs lead:  -0.059
   slope of median total vs lead:   0.686

== tri/R  ell=6  n=237
lead  n  | per-birth boundary reads by colour group: c0   circ(1..m-1)  kill(m)  later(>m) | tota

## Interpretation
1. **The lead slope lives in circulating colours (1..m-1)**:
   - Hex R (loop 3): ~0.95 boundary reads added per lead colour.
   - Square (loop 4): ~0.80 boundary reads added per lead colour.
   - Triangular (loop 6): ~0.65 to 0.35 boundary reads added per lead colour.
   - Hex r (loop 6): ~0.69 to 0.47 boundary reads added per lead colour.
2. **Cancellation by killing and later colours**:
   The slope of reads at the killing colour and later colours is negative (-0.05 to -0.31), pulling down the net slope $b$.
3. **Lag-$\ell$ ring returns do not generalize off the square lattice**:
   On the square lattice, 66–87% of circulating reads return after exactly 4 steps.
   On triangular and hexagonal lattices, lag-$\ell$ share is small (5–44%). The "2x2 ring loop" picture is square-specific.
4. **Hex r rules fill ring centres**:
   15% of lead-1 births on hex r land in cells whose neighbours are all visited (hole births), dropping to 6% at lead 5. Hex R and square sit at 1–2%.

## Caveats and Limitations
- The registered circulation estimator was corrupted by interior loops; passing P3 is an artefact.
- Post hoc colour decomposition restates the identity $1/c = E[N_b]$ and shows *where* the slope lives, but does not yet supply a dynamic mechanism for the plateau survival probability.
- Survivorship selection: kept rules filter out blob/lacy rules, which differ across lattices.

## Next steps
1. Segment boundary-only episodes (runs of consecutive boundary reads of colours 1..m-1) to isolate genuine boundary revisits.
2. Model the per-colour boundary survival plateau as a function of local geometry (e.g. outside unvisited neighbours).